In [9]:
!pip install pandas


[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
from pathlib import Path

csv_path = Path("../data/raw/loan_prediction.csv")
if not csv_path.exists():
    csv_path = Path("../../data/raw/loan_prediction.csv")

df = pd.read_csv(csv_path)

display(df.head());

df.info();


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [11]:
#print unique values appeared in a cloumn
df['Gender'].unique()

array(['Male', 'Female', nan], dtype=object)

In [12]:
#filling with missing values

#Qualitative data
df['Married']=df['Married'].fillna(df['Married'].mode(True)[0])
df['Gender']=df['Gender'].fillna(df['Gender'].mode(True)[0])
df['Dependents']=df['Dependents'].fillna(df['Dependents'].mode(True)[0]) 
df['Self_Employed']=df['Self_Employed'].fillna(df['Self_Employed'].mode(True)[0])
df['Credit_History']=df['Credit_History'].fillna(df['Credit_History'].mode()[0])

#Quantitative data  
df['LoanAmount']=df['LoanAmount'].fillna(df['LoanAmount'].mean())
df['Loan_Amount_Term']=df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mean())

df.isna().sum()

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

In [13]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from sklearn.preprocessing import LabelEncoder

columns = [
    'Gender',
    'Married',
    'Dependents',
    'Education',
    'Self_Employed',
    'Property_Area',
    'Loan_Status'
]

for col in columns:
    le = LabelEncoder()
    le.fit(df[col])
    print(col)
    print(dict(zip(le.classes_, le.transform(le.classes_))))
    print()

Gender
{'Female': np.int64(0), 'Male': np.int64(1)}

Married
{'No': np.int64(0), 'Yes': np.int64(1)}

Dependents
{'0': np.int64(0), '1': np.int64(1), '2': np.int64(2), '3+': np.int64(3)}

Education
{'Graduate': np.int64(0), 'Not Graduate': np.int64(1)}

Self_Employed
{'No': np.int64(0), 'Yes': np.int64(1)}

Property_Area
{'Rural': np.int64(0), 'Semiurban': np.int64(1), 'Urban': np.int64(2)}

Loan_Status
{'N': np.int64(0), 'Y': np.int64(1)}



In [15]:
#splitting the data into training set and testing set
from sklearn.model_selection import train_test_split
X=df.drop(columns=['Loan_ID','Loan_Status'],axis=1)

y=df['Loan_Status']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42)
print(X_train.shape)
print(y_train.shape)

(491, 11)
(491,)


In [ ]:
#we have converted credit history to numerical values since it have only 2 values it have 0 and 1
#for KNN we need to find distance between income and credit hostory - but income have large values shades the credit history values.
#so we need to scale down the values of applicant income column
#now we have already splitted the data to train_set and test set
#to be consistent 
#   -- use .fit_transform() for training data
#   -- use .transform() for testing data // fit_transform also calcualtes the distance and then made splits. So we save our testing data from model learning.
#if you use .fit() on the testing data, the scaler learns the Mean of the hidden exam questions, which is Data Leakage. Your implementation completely avoided this.
from sklearn.preprocessing import StandardScaler
print(X_train.head())
print(X_train.dtypes)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled= scaler.transform(X_test)

ValueError: could not convert string to float: 'Male'

In [ ]:
#Decision Tree
from sklearn.tree import DecisionTreeClassifier

dtc_model = DecisionTreeClassifier()

dtc_model.fit(X_train_scaled,y_train)



,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [ ]:
#test the dtc model now

from sklearn.metrics import accuracy_score

# passing testing data and let the model predict the outputs
y_predict = dtc_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test,y_predict)

print(accuracy)

0.7073170731707317


In [ ]:
#Proof for overfitting
y_predict = dtc_model.predict(X_train_scaled)

accuracy = accuracy_score(y_train,y_predict)

print(accuracy)

1.0


In [ ]:
#RandomForest model

from sklearn.ensemble import RandomForestClassifier

rfc_model = RandomForestClassifier()

rfc_model.fit(X_train_scaled,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
#testing randomforest model

from sklearn.metrics import accuracy_score

Y_predict= rfc_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test,Y_predict)

print(accuracy)

0.7723577235772358


In [ ]:
#KNN model

from sklearn.neighbors import KNeighborsClassifier

knc_model = KNeighborsClassifier()

knc_model.fit(X_train_scaled,y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [ ]:
#testing KNN model
from sklearn.metrics import accuracy_score

y_predict = knc_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test,y_predict)

print(accuracy)

0.7560975609756098


In [ ]:
!pip install Xgboost



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier()

xgb_model.fit(X_train_scaled,y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [ ]:
y_predict = xgb_model.predict(X_test_scaled)

accuracy = accuracy_score(y_test,y_predict)

print(accuracy)

0.7560975609756098


In [ ]:
import pickle

#1. save the model
with open('./loan_approval_model.pkl','wb') as file:
    pickle.dump(xgb_model,file)

#2. save the scalar
with open('./scalar.pkl','wb') as file2:
    pickle.dump(scalar,file2)